# Femora — Breast Cancer Risk Questionnaire (XGBoost)

**Deployed model: BCSC Risk Estimation dataset** (Breast Cancer Surveillance Consortium; Barlow et al., JNCI 2006).
Screening mammograms of women with **no previous breast cancer**. Outcome: breast cancer (invasive or DCIS) diagnosed
**within one year** of the mammogram. The data is aggregated: each row is one combination of risk factors plus a count.

**Reference model: Wisconsin Diagnostic dataset** (Wolberg et al., 1993; the dataset named in the scope document).
569 biopsies described by 30 cell-nucleus measurements from fine-needle aspirate images. It is trained here for comparison,
but it **cannot power a questionnaire**: a user cannot self-report any of its inputs.

**Design decisions**
- **Only self-reportable questions**: age, menopause, BMI, age at first birth, close relatives with breast cancer, previous
  breast biopsy, hormone therapy. Breast density and the last mammogram result are optional. "Don't know" is treated as
  missing, exactly as BCSC codes it.
- **Race/ethnicity excluded**: US census categories don't transfer to Femora's Pakistani users.
- **Monotonic constraints** on established risk factors (age, family history, biopsy, breast density, hormone therapy)
  keep the model medically sensible.
- **No SMOTE / class re-weighting**: the goal is a calibrated absolute risk, not a classifier.
- The output is a 1-year risk **relative to the average woman of the same age**. Bands follow the NICE familial-risk
  categories (moderate ≈ 17%, high ≈ 30% lifetime risk vs 12.5% for the population, so ≈ 1.35× and 2.4×).
- **Symptoms** (lump, nipple discharge, skin changes) are not in any public outcome dataset. The app handles them with
  NICE NG12 referral rules instead of this model.

In [ ]:
import glob, json, os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (accuracy_score, brier_score_loss, f1_score, precision_score, recall_score,
                             roc_auc_score, roc_curve)
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

INPUT = os.environ.get("FEMORA_INPUT", "/kaggle/input")
WORK = os.environ.get("FEMORA_WORK", "/kaggle/working")
OUT = f"{WORK}/model"
os.makedirs(OUT, exist_ok=True)
SEED = 42

## 1. Load the BCSC Risk Estimation dataset

In [ ]:
COLUMNS = ["menopaus", "agegrp", "density", "race", "Hispanic", "bmi", "agefirst", "nrelbc", "brstproc",
           "lastmamm", "surgmeno", "hrt", "invasive", "cancer", "training", "count"]
path = next(p for p in sorted(glob.glob(f"{INPUT}/**/*", recursive=True))
            if os.path.isfile(p) and "risk" in os.path.basename(p).lower() and p.lower().endswith((".txt", ".csv", ".dat")))
with open(path) as f:
    has_header = any(ch.isalpha() for ch in f.readline())
bcsc = pd.read_csv(path, sep=r"[,\s]+", engine="python", header=0 if has_header else None,
                   names=None if has_header else COLUMNS)
bcsc.columns = bcsc.columns.str.strip()

women = bcsc["count"].sum()
cancers = bcsc.loc[bcsc["cancer"] == 1, "count"].sum()
print(f"{path}\n{len(bcsc):,} risk-factor combinations covering {women:,} mammograms; "
      f"{cancers:,} cancers within 1 year ({cancers / women:.3%})")
print(bcsc.groupby("training")["count"].sum().rename({0: "validation", 1: "training"}))

## 2. Features

BCSC codes "unknown" as 9. For hormone therapy and surgical menopause, 9 also means "not menopausal". These become missing
values, which XGBoost handles natively by learning which branch they should follow. Age group 9 is a real value (75–79).

In [ ]:
AGE_GROUPS = {1: "35-39", 2: "40-44", 3: "45-49", 4: "50-54", 5: "55-59", 6: "60-64", 7: "65-69",
              8: "70-74", 9: "75-79", 10: "80-84"}
FEATURES = ["agegrp", "menopaus", "bmi", "agefirst", "nrelbc", "brstproc", "hrt", "surgmeno", "density", "lastmamm"]
MONOTONE = {"agegrp": 1, "menopaus": 0, "bmi": 0, "agefirst": 0, "nrelbc": 1, "brstproc": 1, "hrt": 1,
            "surgmeno": 0, "density": 1, "lastmamm": 0}

X = bcsc[FEATURES].astype(float)
coded = [f for f in FEATURES if f != "agegrp"]
X[coded] = X[coded].mask(X[coded] == 9)
y = bcsc["cancer"].astype(int).values
w = bcsc["count"].astype(float).values
is_train = (bcsc["training"] == 1).values
X_tr, X_va, y_tr, y_va, w_tr, w_va = X[is_train], X[~is_train], y[is_train], y[~is_train], w[is_train], w[~is_train]

def weighted_rate(frame, col):
    g = pd.DataFrame({"level": frame[col].fillna(-1), "w": w, "cases": y * w}).groupby("level").sum()
    return (g["cases"] / g["w"]).rename(index={-1: "unknown"})

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.flat, ["agegrp", "nrelbc", "brstproc", "density", "bmi", "agefirst"]):
    (weighted_rate(X, col) * 1000).plot.bar(ax=ax, color="#C2185B")
    ax.set(title=col, ylabel="cancers per 1,000 women (1 year)", xlabel="")
plt.tight_layout(); plt.show()

## 3. Compare models on the BCSC validation split

The dataset ships with its own training/validation split (`training` column), which is used as-is. Published risk-factor
models (Gail, BCSC) typically reach a ROC-AUC of about 0.60–0.67, so **calibration** (expected ÷ observed cancers ≈ 1)
matters as much as AUC.

In [ ]:
def weighted_metrics(p, y_, w_):
    return {"roc_auc": round(roc_auc_score(y_, p, sample_weight=w_), 4),
            "brier": round(brier_score_loss(y_, p, sample_weight=w_), 6),
            "expected_over_observed": round((p * w_).sum() / (y_ * w_).sum(), 3)}

def one_hot_logit(cols):
    pipe = make_pipeline(OneHotEncoder(handle_unknown="ignore"), LogisticRegression(max_iter=3000))
    pipe.fit(X_tr[cols].fillna(9).astype(int).astype(str), y_tr, logisticregression__sample_weight=w_tr)
    return pipe.predict_proba(X_va[cols].fillna(9).astype(int).astype(str))[:, 1]

fit_rows, stop_rows = train_test_split(np.arange(len(X_tr)), test_size=0.15, stratify=y_tr, random_state=SEED)
model = XGBClassifier(n_estimators=1000, max_depth=4, learning_rate=0.03, subsample=0.8, colsample_bytree=0.9,
                      min_child_weight=20, monotone_constraints=tuple(MONOTONE[f] for f in FEATURES),
                      tree_method="hist", eval_metric="logloss", early_stopping_rounds=50, random_state=SEED)
# early stopping uses a slice of the *training* rows; the BCSC validation split stays untouched
model.fit(X_tr.iloc[fit_rows], y_tr[fit_rows], sample_weight=w_tr[fit_rows],
          eval_set=[(X_tr.iloc[stop_rows], y_tr[stop_rows])], sample_weight_eval_set=[w_tr[stop_rows]], verbose=False)
p_va = model.predict_proba(X_va)[:, 1]
print("trees:", model.best_iteration + 1)

comparison = pd.DataFrame([
    {"model": "Age only (logistic regression)", **weighted_metrics(one_hot_logit(["agegrp"]), y_va, w_va)},
    {"model": "All questions (logistic regression, one-hot)", **weighted_metrics(one_hot_logit(FEATURES), y_va, w_va)},
    {"model": "All questions (XGBoost, monotone) — deployed", **weighted_metrics(p_va, y_va, w_va)},
])
display(comparison)
val_metrics = comparison.iloc[-1].drop("model").to_dict()

cal = pd.DataFrame({"p": p_va, "w": w_va, "cases": y_va * w_va, "pw": p_va * w_va}).sort_values("p")
cal["decile"] = np.minimum((cal["w"].cumsum() / cal["w"].sum() * 10).astype(int), 9)
cal = cal.groupby("decile").sum()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
axes[0].plot(cal["pw"] / cal["w"] * 1000, cal["cases"] / cal["w"] * 1000, "o-", color="#C2185B")
lim = max((cal["pw"] / cal["w"]).max(), (cal["cases"] / cal["w"]).max()) * 1000 * 1.05
axes[0].plot([0, lim], [0, lim], "--", color="grey")
axes[0].set(title="Calibration by risk decile (validation)", xlabel="predicted per 1,000", ylabel="observed per 1,000")
fpr, tpr, _ = roc_curve(y_va, p_va, sample_weight=w_va)
axes[1].plot(fpr, tpr, color="#C2185B"); axes[1].plot([0, 1], [0, 1], "--", color="grey")
axes[1].set(xlabel="False positive rate", ylabel="True positive rate")
axes[1].set_title(f"ROC (AUC = {val_metrics['roc_auc']:.3f})")
plt.tight_layout(); plt.savefig(f"{OUT}/breast_risk_evaluation.png", dpi=150); plt.show()

## 4. Relative risk and risk bands

The app compares a woman's 1-year risk with the **average woman in her age group**, so age alone never makes someone
"high risk". The bands are checked on the validation split: the observed cancer rate should rise from Low to High.

In [ ]:
BANDS = {"low": [0, 1.35], "medium": [1.35, 2.4], "high": [2.4, None]}
p_tr = model.predict_proba(X_tr)[:, 1]
age_avg = (pd.Series(p_tr * w_tr).groupby(X_tr["agegrp"].values).sum()
           / pd.Series(w_tr).groupby(X_tr["agegrp"].values).sum())

expected_va = X_va["agegrp"].map(age_avg).values
rr_va = p_va / expected_va
band_va = np.select([rr_va < 1.35, rr_va < 2.4], ["low", "medium"], "high")
band_table = (pd.DataFrame({"band": band_va, "women": w_va, "cases": y_va * w_va, "expected_at_age_average": expected_va * w_va})
              .groupby("band").sum().reindex(["low", "medium", "high"]).fillna(0))
band_table["share_of_women"] = band_table["women"] / band_table["women"].sum()
band_table["observed_per_1000"] = band_table["cases"] / band_table["women"] * 1000
band_table["observed_rr_vs_age_average"] = band_table["cases"] / band_table["expected_at_age_average"]
display(band_table.round(3))
print((age_avg * 1000).round(2).rename(AGE_GROUPS).rename("average 1-year risk per 1,000"))

## 5. Sanity checks & feature importance

In [ ]:
profiles = pd.DataFrame([
    {"name": "42, no risk factors",            "agegrp": 2, "menopaus": 0, "bmi": 1, "agefirst": 0, "nrelbc": 0, "brstproc": 0},
    {"name": "42, mother had breast cancer",   "agegrp": 2, "menopaus": 0, "bmi": 1, "agefirst": 0, "nrelbc": 1, "brstproc": 0},
    {"name": "42, 2 relatives + prior biopsy", "agegrp": 2, "menopaus": 0, "bmi": 1, "agefirst": 2, "nrelbc": 2, "brstproc": 1},
    {"name": "58, post-menopausal, BMI 32, HRT", "agegrp": 5, "menopaus": 1, "bmi": 3, "agefirst": 1, "nrelbc": 0,
     "brstproc": 0, "hrt": 1, "surgmeno": 0},
]).set_index("name").reindex(columns=FEATURES).astype(float)
p_prof = model.predict_proba(profiles)[:, 1]
print(pd.DataFrame({"1-year risk per 1,000": p_prof * 1000,
                    "x age average": p_prof / profiles["agegrp"].map(age_avg).values}, index=profiles.index).round(2))

imp = pd.Series(model.get_booster().get_score(importance_type="total_gain")).reindex(FEATURES).fillna(0)
imp = imp / imp.sum()
fig, ax = plt.subplots(figsize=(8, 4.5))
imp.sort_values().plot.barh(ax=ax, color="#C2185B"); ax.set_title("XGBoost feature importance (share of total gain)")
plt.tight_layout(); plt.savefig(f"{OUT}/breast_risk_feature_importance.png", dpi=150); plt.show()

## 6. Reference: XGBoost on the Wisconsin Diagnostic dataset

This is the dataset named in the scope document. It scores very well, but every input is a microscope measurement of cells
taken by a needle biopsy (radius, texture, concavity...). It answers "is this biopsied lump malignant?", not "what is
my risk?". That's why it is kept as a reference and the app uses the BCSC model.

In [ ]:
wisconsin_files = glob.glob(f"{INPUT}/**/data.csv", recursive=True)
if wisconsin_files:
    wis = pd.read_csv(wisconsin_files[0]).drop(columns=["id", "Unnamed: 32"], errors="ignore")
    Xw, yw = wis.drop(columns="diagnosis"), (wis["diagnosis"] == "M").astype(int)
else:   # the same data ships with scikit-learn (there 0 = malignant)
    data = load_breast_cancer(as_frame=True)
    Xw, yw = data.data, 1 - data.target

wis_model = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9,
                          eval_metric="logloss", random_state=SEED)
scores = cross_validate(wis_model, Xw, yw, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                        scoring=["accuracy", "f1", "recall", "roc_auc"])
wisconsin_cv = {m: f"{scores['test_' + m].mean():.3f} ± {scores['test_' + m].std():.3f}"
                for m in ["accuracy", "f1", "recall", "roc_auc"]}
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(Xw, yw, test_size=0.2, stratify=yw, random_state=SEED)
wis_model.fit(Xw_tr, yw_tr)
pw = wis_model.predict_proba(Xw_te)[:, 1]
wisconsin_test = {"test_size": int(len(yw_te)), "accuracy": round(accuracy_score(yw_te, pw >= 0.5), 4),
                  "precision": round(precision_score(yw_te, pw >= 0.5), 4), "recall": round(recall_score(yw_te, pw >= 0.5), 4),
                  "f1": round(f1_score(yw_te, pw >= 0.5), 4), "roc_auc": round(roc_auc_score(yw_te, pw), 4)}
print("5-fold CV:", wisconsin_cv)
print("held-out 20%:", wisconsin_test)
print("top inputs:", list(pd.Series(wis_model.feature_importances_, index=Xw.columns).nlargest(5).index))

## 7. Export for the Femora backend

In [ ]:
model.save_model(f"{OUT}/breast_risk_xgb.json")

def to_builtin(o):
    return o.item() if hasattr(o, "item") else str(o)

with open(f"{OUT}/breast_risk_meta.json", "w") as f:
    json.dump({
        "features": FEATURES,
        "monotone_constraints": MONOTONE,
        "age_groups": {str(k): v for k, v in AGE_GROUPS.items()},
        "age_average_risk": {str(int(k)): float(v) for k, v in age_avg.items()},
        "relative_risk_bands": BANDS,
        "validation_metrics": val_metrics,
        "comparison": comparison.to_dict(orient="records"),
        "band_validation": band_table.reset_index().to_dict(orient="records"),
        "feature_importance": imp.sort_values(ascending=False).round(4).to_dict(),
        "dataset": {"name": "BCSC Risk Estimation Dataset (Barlow et al., JNCI 2006)", "file": os.path.basename(path),
                    "combinations": int(len(bcsc)), "mammograms": int(women), "cancers_1yr": int(cancers)},
        "wisconsin_reference": {
            "dataset": "Breast Cancer Wisconsin (Diagnostic), Wolberg et al. 1993 (n=569)",
            "cv": wisconsin_cv, "test": wisconsin_test,
            "note": "Inputs are cytology measurements from a biopsy, so it is not usable as a self-report questionnaire.",
        },
    }, f, indent=2, default=to_builtin)
print(sorted(os.listdir(OUT)))